
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>


# Demo: Setting Up and Managing LakeFlow Jobs using UI

In this demo, we'll set up a Databricks Lakeflow Jobs that automates a series of MLOps tasks such as data quality assessment, feature importance analysis, and alerting on unusual patterns. These tasks help ensure data readiness and provide insights before moving to model training. We’ll also introduce a conditional path based on the detection of unusual patterns.

**Learning Objectives:**

In this demo, we will:

- Create and configure a Databricks Lakeflow job with multiple Python script tasks.
- Set dependencies and conditional paths between tasks.
- Enable email notifications for successful job runs.
- Manually trigger the workflow.
- Monitor the job's execution and completion.

## REQUIRED - SELECT CLASSIC COMPUTE
Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.
Follow these steps to select the classic compute cluster:
1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.
1. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:
   - In the drop-down, select **More**.
   - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.
  
**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:
1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.
1. Find the triangle icon to the right of your compute cluster name and click it.
1. Wait a few minutes for the cluster to start.
1. Once the cluster is running, complete the steps above to select your cluster.

## Requirements

Please review the following requirements before starting the lesson:

- To run this notebook, you need to use one of the following Databricks runtime(s): `16.3.x-cpu-ml-scala2.12`

## Optional: Using a Serverless Cluster for This Notebook

Instructors have the flexibility to use a **Serverless cluster** for this notebook if preferred. Serverless clusters can provide faster startup times and simplified resource management, making them a convenient option for running the notebook efficiently.

## Task 1: Create a Databricks LakeFlow Jobs in the UI

1. **Navigate to Jobs & Pipelines**:
   - In your Databricks workspace, click on the **Jobs & Pipelines** icon in the left sidebar.
   
2. **Create a New Job**:
   - Click on **Create** in the upper-right corner of the Jobs & Pipelines page and Select **Job**.
   - Name the job "MLOps Workflow: Data Quality and Feature Analysis" or something similar for easy identification.

## Task 2: Add Tasks to the Job:

### Task 2.1: Data Quality Assessment

1. **Create First Task**:
   - Name the task `Data_Quality_Assessment`.
   - Set **Type** to `Notebook`.
   - **Source** should be set to `Workspace`.
   - Set **Path** to the notebook for data quality assessment (e.g., `/1.1 Demo Pipeline - Data Quality and Feature Analysis/1.1a - Data Quality Assessment`).
   - Select an `Serverless` cluster for this task.
   - Click **Create Task**.

This task will check for missing values, duplicates, and outliers in the dataset and generate a data quality report.

### Task 2.2: Alert on Unusual Patterns

1. **Create Second Task**:
   - Click on **Add Task --> Notebook**.
   - Name the task `Alert_Unusual_Patterns`.
   - Set **Type** to `Notebook`.
   - **Source** should be set to `Workspace`.
   - Set **Path** to the notebook for alerting on unusual patterns (e.g., `/1.1 Demo Pipeline - Data Quality and Feature Analysis/1.1b - Alert on Unusual Patterns`).
   - Use the same `Serverless` cluster as the previous task.
   - Set **Depends on** to `Data_Quality_Assessment` to ensure this task runs after data quality checks.
   - Click **Create Task**.

This task will check for unusual patterns, such as high cardinality and skewed distributions, setting a flag if any unusual patterns are detected.

### Task 2.3: Conditional Path Setup

1. **Create Condition for Unusual Patterns**:
   - Click on **Add Task --> If/else condition.**
   - Name the condition task `Alert_Unusual_Patterns_True`.
   - Set **Type** to `If/else condition`.
   - **Condition**: Set the expression to **&lcub;&lcub;tasks.Alert_Unusual_Patterns.values.unusual_pattern_status&rcub;&rcub; == unusual_pattern_detected**.
   - Set **Depends on** to `Alert_Unusual_Patterns` to ensure this condition is evaluated after the unusual patterns check.
   - This condition will branch based on whether unusual patterns are detected (`True`) or not (`False`).
   - Click **Save Task**.

This conditional setup directs the workflow to either investigate unusual patterns (if detected) or proceed with feature importance analysis if no patterns are detected.

### Task 2.4: Investigate and Resolve Unusual Patterns and Analyse Feature Importance

1. **Create Third Task**:
   - Click on **Add Task --> Notebook**.
   - Name the task `Investigate_Unusual_Patterns`.
   - Set **Type** to `Notebook`.
   - **Source** should be set to `Workspace`.
   - Set **Path** to the notebook for investigating unusual patterns (e.g., `/1.1 Demo Pipeline - Data Quality and Feature Analysis/1.1c -Investigate and Resolve Unusual Patterns`).
   - Use the same cluster as the previous tasks.
   - Set **Depends on** to `Alert_Unusual_Patterns_True(true)`, ensuring it only runs if unusual patterns are detected.
   - Click **Create Task**.

This task will execute only if unusual patterns are detected (when the condition is `True`).

### Task 2.5: Feature Importance Analysis (False Path or After Investigation)

1. **Create Fourth Task**:
   - Click on **Add Task --> Notebook**.
   - Name the task `Feature_Importance`.
   - Set **Type** to `Notebook`.
   - **Source** should be set to `Workspace`.
   - Set **Path** to the notebook for feature importance analysis (e.g., `/1.1 Demo Pipeline - Data Quality and Feature Analysis/1.1d - Feature Importance Analysis`).
   - Use the same cluster as the previous tasks.
   - Set **Depends on** to:
     - `Alert_Unusual_Patterns_True(false)` (to run if no unusual patterns are detected).
   - Click **Create Task**.

This task will run only if there are no unusual patterns or after the unusual patterns investigation is completed.

### Task 2.6: Save Report (Final Task)

1. **Create Fifth Task**:
   - Click on **Add Task --> Notebook**.
   - Name the task `Save_Report`.
   - Set **Type** to `Notebook`.
   - **Source** should be set to `Workspace`.
   - Set **Path** to the notebook for saving the final report (e.g., `/1.1 Demo Pipeline - Data Quality and Feature Analysis/1.1e - Save Report Notebook (Success Path)`).
   - Use the same cluster as the previous tasks.
   - Set **Depends on** to both:
     - `Feature_Importance` and `Investigate_Unusual_Patterns`.
   - Set **Run if dependencies** to "At least one succeeded" to ensure it saves the report regardless of the path taken.
   - Click **Create Task**.

This task will save the final report once all prior steps are successfully completed, regardless of whether unusual patterns were detected.

### Task 2.7: Enable Email Notifications

1. **Set up Notifications**:
   - In the job's configuration, navigate to the **Job Notifications** section.
   - Enable email notifications by adding your email to receive updates on job completion.

## Task 3: Trigger the Job Manually

1. **Run the Job**:
   - Go to the job in the Databricks UI and click on **Run Now** in the top-right corner to manually trigger the job. This will execute all tasks in the Job according to their dependencies and conditions.

## Task 4: Monitor the Job Execution

1. **Navigate to the Runs Tab**:
   - In the job interface, go to the **Runs** tab to view active and completed executions of the job.

2. **Observe Task Execution**:
   - Each task’s status is displayed in the **Runs** tab, where you can see which tasks are currently executing or have completed.
   - Click on each task to view its execution details and outputs, allowing you to troubleshoot and verify each stage.
   - Check the logs to see if the Job followed the correct path based on the unusual pattern detection condition.

## Conclusion

In this demo, you learned how to:
- Configure and execute a Databricks LakeFlow Jobs with multiple tasks for data quality, feature importance, and unusual pattern checks.
- Use dependencies and conditional paths to control the flow of tasks based on the data conditions.
- Set up email notifications to stay updated on job execution.
- Trigger the Job manually and monitor its execution.

This workflow serves as a preliminary step to ensure data quality and feature insights before moving on to model training. By automating these MLOps setup tasks and handling conditional paths, you can ensure a robust pipeline that adapts based on data characteristics, providing insights and addressing issues early in the MLOps process.


&copy; 2025 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="blank">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy" target="blank">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use" target="blank">Terms of Use</a> | 
<a href="https://help.databricks.com/" target="blank">Support</a>